In [24]:
import pandas as pd
import numpy as np
import itertools
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 2137

In [25]:
df = pd.read_csv("data_for_classifier.csv")

In [26]:
df = df.drop(columns = "datetime")

In [27]:
split_idx = int(len(df) * 0.9)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

X_train = train.drop(columns = "is_fraud")
y_train = train["is_fraud"]

X_test = test.drop(columns = "is_fraud")
y_test = test["is_fraud"]

In [28]:
val_split_idx = int(len(train) * (8/9)) 

X_train_sub = X_train[:val_split_idx]
y_train_sub = y_train[:val_split_idx]

X_val = X_train[val_split_idx:]
y_val = y_train[val_split_idx:]

In [7]:
learning_rates = [0.01, 0.05, 0.1]
max_iters = [100, 200, 500]
max_leaf_nodes = [15, 31, 63]

best_f1 = 0
best_params = {}

for lr, mi, mln in itertools.product(learning_rates, max_iters, max_leaf_nodes):
    gbc = HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        learning_rate=lr,
        max_iter=mi,
        max_leaf_nodes=mln,
        class_weight="balanced"
    )
    gbc.fit(X_train_sub, y_train_sub)
    y_pred = gbc.predict(X_val)
    f1 = f1_score(y_val, y_pred, pos_label=1)
    if f1 > best_f1:
        best_f1 = f1
        best_params = {"learning_rate" : lr, "max_iter" : mi, "max_leaf_nodes" : mln}

print(f"Best f1 score: {best_f1}")

Best f1 score: 0.8487084870848709


In [30]:
gbc = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    learning_rate=best_params["learning_rate"],
    max_iter=best_params["max_iter"],
    max_leaf_nodes=best_params["max_leaf_nodes"],
    class_weight="balanced"
)
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.87      0.95      0.91       185

    accuracy                           1.00    109018
   macro avg       0.93      0.98      0.95    109018
weighted avg       1.00      1.00      1.00    109018



Recall is more important as high recall means we rarely miss a laundering case (false negative).

In [31]:
result = permutation_importance(gbc, X_test, y_test, n_jobs=-1)
importance_df = pd.DataFrame({"Feature": X_test.columns, "Importance": result.importances_mean})
importance_df = importance_df.sort_values(by="Importance", ascending=True)
importance_df.to_csv("importance_features_HGBC.csv", index=False)

/Users/mezzylane/Desktop/Lang/Aalto/CS_C3250_DSP/dsp-op/money laundering/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [32]:
features_to_keep = importance_df[importance_df["Importance"] >= 0]["Feature"].tolist()

X_train = X_train[features_to_keep]
X_test = X_test[features_to_keep]

print(f"New X_train shape: {X_train.shape}")
print(f"New X_test shape: {X_test.shape}")

New X_train shape: (981154, 67)
New X_test shape: (109018, 67)


In [33]:
gbc = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    learning_rate=best_params["learning_rate"],
    max_iter=best_params["max_iter"],
    max_leaf_nodes=best_params["max_leaf_nodes"],
    class_weight="balanced"
)
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.83      0.93      0.88       185

    accuracy                           1.00    109018
   macro avg       0.91      0.96      0.94    109018
weighted avg       1.00      1.00      1.00    109018

